# Small-Signal Stability Analysis: Eigenvalue-Based Small-Signal Stability## ObjectiveDetermines the impact of new generation on system oscillatory modesProject: 39 Bus New England System - 2**Study Case: Study Cases 1. Power Flow****Objective:**- Determine the impact of new generation on system oscillatory modes- Perform eigenvalue analysis for Base Case and New Generation Case- Identify critical modes and damping ratios- Outputs: Eigenvalue plots, Damping ratio comparison---

## Step 1: Access PowerFactoryFirst, we need to set up the Python environment to access DIgSILENT PowerFactory.

In [ ]:
# ============================================================================# STEP 1: Access PowerFactory# ============================================================================import osos.environ["PATH"] = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2" + os.environ["PATH"]import syssys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9")# Import powerfactoryimport powerfactory as pfapp = pf.GetApplication()  # Get the application# ============================================================================# STEP 2: Access and activate project# ============================================================================user = app.GetCurrentUser()project = app.ActivateProject("39 Bus New England System - 2")  # Activate the desired projectprj = app.GetActiveProject()print(f"Project activated: {prj.loc_name}")# ============================================================================# STEP 2.5: Activate study case (if needed)# ============================================================================# Try to activate the study case "Study Cases 1. Power Flow"try:    study_cases = prj.GetContents('*.IntCase')    for sc in study_cases:        if '1. Power Flow' in sc.loc_name or 'Power Flow' in sc.loc_name:            sc.Activate()            print(f"Study case activated: {sc.loc_name}")            breakexcept:    print("Note: Using default/active study case")# ============================================================================# STEP 3: Set initial conditions - Base Case# ============================================================================print("\n=== Setting Initial Conditions - Base Case ===")app.ResetCalculation()# Set initial conditionsini = app.GetFromStudyCase('ComInc')ini.Execute()print("Initial conditions calculated for Base Case")# ============================================================================# STEP 4: Run Eigenvalue Analysis - Base Case# ============================================================================print("\n=== Running Eigenvalue Analysis - Base Case ===")try:    # Get eigenvalue analysis command    eigenval = app.GetFromStudyCase('ComEig')        # Configure eigenvalue analysis    eigenval.iopt_net = 0  # Balanced 3-phase calculation    eigenval.iopt_eig = 0  # All eigenvalues        # Execute eigenvalue calculation    eigenval.Execute()        print("Eigenvalue calculation completed for Base Case")        # Get eigenvalue results    eigen_results_base = []    eigen_objects = app.GetCalcRelevantObjects('*.IntEig')        for eig_obj in eigen_objects:        try:            real_part = eig_obj.GetAttribute('real')            imag_part = eig_obj.GetAttribute('imag')            freq = eig_obj.GetAttribute('freq')            damping = eig_obj.GetAttribute('damp')            damping_ratio = eig_obj.GetAttribute('damp_ratio')                        eigen_results_base.append({                'real': real_part,                'imag': imag_part,                'frequency_Hz': freq,                'damping': damping,                'damping_ratio': damping_ratio            })        except:            continue        print(f"Extracted {len(eigen_results_base)} eigenvalues for Base Case")    except Exception as e:    print(f"Note: Eigenvalue analysis may need manual setup: {e}")    print("Eigenvalue analysis requires proper system linearization")    eigen_results_base = []# ============================================================================# STEP 5: Export Base Case Eigenvalue Results# ============================================================================import csvimport osscript_dir = os.path.dirname(os.path.abspath(__file__))if eigen_results_base:    base_eigen_csv_path = os.path.join(script_dir, 'eigenvalue_results_base_case.csv')    with open(base_eigen_csv_path, 'w', newline='') as csvfile:        writer = csv.writer(csvfile)        writer.writerow(['Real Part', 'Imaginary Part', 'Frequency (Hz)', 'Damping', 'Damping Ratio'])        for eig in eigen_results_base:            writer.writerow([eig['real'], eig['imag'], eig['frequency_Hz'],                            eig['damping'], eig['damping_ratio']])        print(f"Base case eigenvalue results exported to: eigenvalue_results_base_case.csv")# ============================================================================# STEP 6: New Generation Case Eigenvalue Analysis# ============================================================================print("\n=== Running Eigenvalue Analysis - New Generation Case ===")# NOTE: This section should be modified based on how new generation is addedapp.ResetCalculation()# Set initial conditionsini.Execute()print("Initial conditions calculated for New Generation Case")try:    eigenval = app.GetFromStudyCase('ComEig')    eigenval.iopt_net = 0    eigenval.iopt_eig = 0    eigenval.Execute()        print("Eigenvalue calculation completed for New Generation Case")        eigen_results_new_gen = []    eigen_objects = app.GetCalcRelevantObjects('*.IntEig')        for eig_obj in eigen_objects:        try:            real_part = eig_obj.GetAttribute('real')            imag_part = eig_obj.GetAttribute('imag')            freq = eig_obj.GetAttribute('freq')            damping = eig_obj.GetAttribute('damp')            damping_ratio = eig_obj.GetAttribute('damp_ratio')                        eigen_results_new_gen.append({                'real': real_part,                'imag': imag_part,                'frequency_Hz': freq,                'damping': damping,                'damping_ratio': damping_ratio            })        except:            continue        print(f"Extracted {len(eigen_results_new_gen)} eigenvalues for New Generation Case")    except Exception as e:    print(f"Note: Eigenvalue analysis may need manual setup: {e}")    eigen_results_new_gen = []# ============================================================================# STEP 7: Export New Generation Case Eigenvalue Results# ============================================================================if eigen_results_new_gen:    new_gen_eigen_csv_path = os.path.join(script_dir, 'eigenvalue_results_new_gen_case.csv')    with open(new_gen_eigen_csv_path, 'w', newline='') as csvfile:        writer = csv.writer(csvfile)        writer.writerow(['Real Part', 'Imaginary Part', 'Frequency (Hz)', 'Damping', 'Damping Ratio'])        for eig in eigen_results_new_gen:            writer.writerow([eig['real'], eig['imag'], eig['frequency_Hz'],                            eig['damping'], eig['damping_ratio']])        print(f"New generation case eigenvalue results exported to: eigenvalue_results_new_gen_case.csv")# ============================================================================# STEP 8: Load CSV Data and Create Visualizations# ============================================================================print("\n=== Creating Visualizations ===")try:    import pandas as pd    import matplotlib.pyplot as plt    import seaborn as sns    import numpy as np    from bokeh.plotting import figure, output_file, save    from bokeh.models import ColumnDataSource, HoverTool    from bokeh.layouts import gridplot        # Set style    sns.set_style("whitegrid")    plt.rcParams['figure.figsize'] = (16, 10)        # Load CSV data if available    if eigen_results_base and eigen_results_new_gen:        base_df = pd.read_csv(base_eigen_csv_path) if os.path.exists(base_eigen_csv_path) else pd.DataFrame(eigen_results_base)        new_gen_df = pd.read_csv(new_gen_eigen_csv_path) if os.path.exists(new_gen_eigen_csv_path) else pd.DataFrame(eigen_results_new_gen)                # Create visualizations        fig, axes = plt.subplots(2, 2, figsize=(16, 12))                # 1. Eigenvalue plot (s-plane)        base_real = base_df['Real Part'].values if 'Real Part' in base_df.columns else [e['real'] for e in eigen_results_base]        base_imag = base_df['Imaginary Part'].values if 'Imaginary Part' in base_df.columns else [e['imag'] for e in eigen_results_base]        new_gen_real = new_gen_df['Real Part'].values if 'Real Part' in new_gen_df.columns else [e['real'] for e in eigen_results_new_gen]        new_gen_imag = new_gen_df['Imaginary Part'].values if 'Imaginary Part' in new_gen_df.columns else [e['imag'] for e in eigen_results_new_gen]                axes[0, 0].scatter(base_real, base_imag, c='blue', marker='o', s=60,                           label='Base Case', alpha=0.7, edgecolors='black', linewidths=0.5)        axes[0, 0].scatter(new_gen_real, new_gen_imag, c='red', marker='s', s=60,                           label='New Generation Case', alpha=0.7, edgecolors='black', linewidths=0.5)        axes[0, 0].axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)        axes[0, 0].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)        axes[0, 0].set_xlabel('Real Part (1/s)', fontsize=12)        axes[0, 0].set_ylabel('Imaginary Part (rad/s)', fontsize=12)        axes[0, 0].set_title('Eigenvalue Comparison (s-plane)', fontsize=12, fontweight='bold')        axes[0, 0].legend()        axes[0, 0].grid(True, alpha=0.3)                # 2. Damping ratio comparison        base_damping = base_df['Damping Ratio'].abs().values if 'Damping Ratio' in base_df.columns else [abs(e.get('damping_ratio', 0)) for e in eigen_results_base if e.get('damping_ratio') is not None]        new_gen_damping = new_gen_df['Damping Ratio'].abs().values if 'Damping Ratio' in new_gen_df.columns else [abs(e.get('damping_ratio', 0)) for e in eigen_results_new_gen if e.get('damping_ratio') is not None]                if len(base_damping) > 0 and len(new_gen_damping) > 0:            axes[0, 1].hist(base_damping, bins=20, alpha=0.7, label='Base Case', color='blue', edgecolor='black')            axes[0, 1].hist(new_gen_damping, bins=20, alpha=0.7, label='New Generation Case', color='red', edgecolor='black')            axes[0, 1].set_xlabel('Damping Ratio', fontsize=12)            axes[0, 1].set_ylabel('Number of Modes', fontsize=12)            axes[0, 1].set_title('Damping Ratio Distribution', fontsize=12, fontweight='bold')            axes[0, 1].legend()            axes[0, 1].grid(True, alpha=0.3, axis='y')                # 3. Frequency distribution        base_freq = base_df['Frequency (Hz)'].abs().values if 'Frequency (Hz)' in base_df.columns else [abs(e.get('frequency_Hz', 0)) for e in eigen_results_base]        new_gen_freq = new_gen_df['Frequency (Hz)'].abs().values if 'Frequency (Hz)' in new_gen_df.columns else [abs(e.get('frequency_Hz', 0)) for e in eigen_results_new_gen]                axes[1, 0].hist(base_freq, bins=30, alpha=0.7, label='Base Case', color='blue', edgecolor='black')        axes[1, 0].hist(new_gen_freq, bins=30, alpha=0.7, label='New Generation Case', color='red', edgecolor='black')        axes[1, 0].set_xlabel('Frequency (Hz)', fontsize=12)        axes[1, 0].set_ylabel('Number of Modes', fontsize=12)        axes[1, 0].set_title('Oscillation Frequency Distribution', fontsize=12, fontweight='bold')        axes[1, 0].legend()        axes[1, 0].grid(True, alpha=0.3, axis='y')                # 4. Real part vs Frequency        axes[1, 1].scatter(base_freq, base_real, c='blue', marker='o', s=60,                           label='Base Case', alpha=0.7, edgecolors='black', linewidths=0.5)        axes[1, 1].scatter(new_gen_freq, new_gen_real, c='red', marker='s', s=60,                           label='New Generation Case', alpha=0.7, edgecolors='black', linewidths=0.5)        axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)        axes[1, 1].set_xlabel('Frequency (Hz)', fontsize=12)        axes[1, 1].set_ylabel('Real Part (1/s)', fontsize=12)        axes[1, 1].set_title('Real Part vs Frequency', fontsize=12, fontweight='bold')        axes[1, 1].legend()        axes[1, 1].grid(True, alpha=0.3)                plt.tight_layout()        eigen_plot_path = os.path.join(script_dir, 'eigenvalue_analysis_plots.png')        plt.savefig(eigen_plot_path, dpi=300, bbox_inches='tight')        plt.close()        print(f"Static plots saved to: eigenvalue_analysis_plots.png")                # Interactive Bokeh plot        try:            output_file(os.path.join(script_dir, 'eigenvalue_interactive.html'))                        p1 = figure(width=900, height=500, title="Eigenvalue Comparison (Interactive s-plane)",                       x_axis_label="Real Part (1/s)", y_axis_label="Imaginary Part (rad/s)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        base_source = ColumnDataSource(data=dict(                x=base_real,                y=base_imag,                freq=base_freq[:len(base_real)] if len(base_freq) >= len(base_real) else base_freq,                damping=base_damping[:len(base_real)] if len(base_damping) >= len(base_real) else base_damping            ))                        new_gen_source = ColumnDataSource(data=dict(                x=new_gen_real,                y=new_gen_imag,                freq=new_gen_freq[:len(new_gen_real)] if len(new_gen_freq) >= len(new_gen_real) else new_gen_freq,                damping=new_gen_damping[:len(new_gen_real)] if len(new_gen_damping) >= len(new_gen_real) else new_gen_damping            ))                        p1.circle('x', 'y', size=8, source=base_source, color='blue', alpha=0.7, legend_label='Base Case')            p1.square('x', 'y', size=8, source=new_gen_source, color='red', alpha=0.7, legend_label='New Generation Case')            p1.line([0, 0], [min(min(base_imag), min(new_gen_imag)), max(max(base_imag), max(new_gen_imag))],                    color='black', line_dash='dashed', line_width=2)                        hover = p1.select_one(HoverTool)            hover.tooltips = [("Real", "@x{0.00}"), ("Imag", "@y{0.00}"),                              ("Freq", "@freq{0.00} Hz"), ("Damping", "@damping{0.00}")]                        p1.legend.location = "top_right"                        # Damping ratio comparison            p2 = figure(width=900, height=400, title="Damping Ratio Comparison (Interactive)",                       x_axis_label="Damping Ratio", y_axis_label="Frequency",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        p2.circle(base_damping[:len(base_freq)], base_freq[:len(base_damping)],                      size=8, color='blue', alpha=0.7, legend_label='Base Case')            p2.square(new_gen_damping[:len(new_gen_freq)], new_gen_freq[:len(new_gen_damping)],                      size=8, color='red', alpha=0.7, legend_label='New Generation Case')                        hover2 = p2.select_one(HoverTool)            hover2.tooltips = [("Damping", "@x{0.00}"), ("Frequency", "@y{0.00} Hz")]                        grid = gridplot([[p1], [p2]], toolbar_location='right')            save(grid)            print(f"Interactive plots saved to: eigenvalue_interactive.html")        except Exception as e:            print(f"Note: Bokeh interactive plot creation failed: {e}")    except ImportError as e:    print(f"Note: Visualization libraries not available: {e}")    print("Install required packages: pip install matplotlib seaborn pandas bokeh numpy")except Exception as e:    print(f"Note: Error creating visualizations: {e}")# ============================================================================# STEP 9: Clean up# ============================================================================app.ResetCalculation()print("\n=== Eigenvalue Analysis completed successfully ===")print(f"Results saved in: {script_dir}")